In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, LongType, StringType
import pandas as pd

1. dim_date

In [0]:
# 1. Recreate the business-day range (2000-2019)
date_range = pd.date_range(start="2000-01-01", end="2019-12-31", freq="B")
date_df = pd.DataFrame({"Full_Date": date_range})

# 2. Convert to Spark, cast explicitly, enrich with sorting numbers, and order columns
dim_date_spark = spark.createDataFrame(date_df)
dim_date_spark = (
    dim_date_spark
    .withColumn("Date_Key", F.date_format("Full_Date", "yyyyMMdd").cast(LongType()))
    .withColumn("Full_Date", F.col("Full_Date").cast(DateType()))
    .withColumn("Day_of_Week", F.date_format("Full_Date", "EEEE"))
    .withColumn("Day_Number_Of_Week", F.dayofweek("Full_Date")) # For custom weekday sorting
    .withColumn("Month_Name", F.date_format("Full_Date", "MMMM"))
    .withColumn("Month_Number", F.month("Full_Date")) # For chronological month sorting
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Full_Date")))
    .withColumn("Year", F.year("Full_Date"))
)

# 3. Write to Gold layer
(
    dim_date_spark.write.format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_date")
)

print("Successfully created and populated gold.dim_date!")

2. dim_region

In [0]:
# 1. Define region metadata
regions_data = [
    (1, "USA", "United States", "S&P 500", "USD"),
    (2, "EMU", "Euro Area", "STOXX Europe 50", "EUR"),
    (3, "CHN", "China", "Shanghai Composite", "CNY"),
    (4, "JPN", "Japan", "Nikkei 225", "JPY"),
    (5, "IND", "India", "Nifty 50", "INR"),
]

columns = [
    "Region_Key",
    "Country_Code",
    "Region_Name",
    "Stock_Exchange",
    "Currency_Code",
]

# 2. Create Spark DataFrame
dim_region_spark = spark.createDataFrame(regions_data, columns)

# 3. Write to Gold layer
(
    dim_region_spark.write.format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_region")
)

print("Successfully created and populated gold.dim_region!")